In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

%load_ext autoreload
%autoreload 2

/home/turbotowerlnx/Documents/Master/TLH/TLH-CharoCharito-Alexa/notebooks
/home/turbotowerlnx/Documents/Master/TLH/TLH-CharoCharito-Alexa/app


In [2]:
import os
from langchain_ollama import OllamaLLM, OllamaEmbeddings
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

/home/turbotowerlnx/Documents/Master/TLH/TLH-CharoCharito-Alexa/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from src.config import Configuration
CONFIG = Configuration()

In [4]:
def cargar_y_indexar(CONFIG: Configuration):
    embeddings = OllamaEmbeddings(model=CONFIG.rag_embedding_model)

    if os.path.exists(CONFIG.db_path):
        print("--- Cargando base de datos existente... ---")
        return Chroma(persist_directory=CONFIG.db_path, embedding_function=embeddings)
    print(f"--- Escaneando PDFs en {CONFIG.pdf_path} ---")
    
    # DirectoryLoader busca todos los archivos .pdf y usa PyPDFLoader para cada uno
    loader = DirectoryLoader(
        CONFIG.pdf_path,
        glob="./*.pdf",
        loader_cls=PyPDFLoader
    )
    
    docs = loader.load()
    print(f"Documentos cargados: {len(docs)} páginas encontradas.")

    # Dividimos el texto en trozos
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = text_splitter.split_documents(docs)

    print(f"Creando base de datos vectorial con {len(splits)} fragmentos...")
    vectorstore = Chroma.from_documents(
        documents=splits, 
        embedding=embeddings, 
        persist_directory=CONFIG.db_path
    )
    return vectorstore

In [5]:
vectorstore = cargar_y_indexar(CONFIG)
llm = OllamaLLM(model=CONFIG.rag_model_name)

--- Cargando base de datos existente... ---


/tmp/ipykernel_263364/1246956600.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(persist_directory=CONFIG.db_path, embedding_function=embeddings)


In [ ]:
system_prompt = (
    "Eres un asistente de investigación. Responde basándote estrictamente en el contexto."
    "IMPORTANTE: Al final de tu respuesta, indica el nombre del archivo y la página de donde "
    "proviene la información (esta info está en los metadatos 'source' y 'page')."
    "\n\n"
    "Contexto: {context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
    ])

In [7]:
chain = create_retrieval_chain(
    vectorstore.as_retriever(search_kwargs={"k": 10}), 
    create_stuff_documents_chain(llm, prompt)
)

In [8]:
query = input("\nPregunta sobre tus documentos: ")

print("Buscando en la biblioteca...")
res = chain.invoke({"input": query})
print(f"\nRespuesta:\n{res['answer']}")

Buscando en la biblioteca...

Respuesta:
Las principales métricas de resumen automático son:

1. **ROUGE (Recall-Oriented Understudy for Gisting Evaluation)** (Lin, 2004): Esta métrica estándar mide el recuerdo (recall), es decir, cuánta información del resumen humano ha sido capturada por la máquina.
2. **BERTScore** (T. Zhang et al., 2020): Calcula la similitud semántica mediante embeddings, superando la limitación de buscar coincidencias exactas de palabras.
3. **BARTScore** (Yuan et al., 2021): Evalúa la calidad basándose en la probabilidad de generación de un modelo de lenguaje, puntuando de forma integral la coherencia, la fluidez y la fidelidad del resumen.

Estas métricas se utilizan para evaluar la calidad de los resúmenes automáticos y compararlos con los resúmenes humanos. Cada una tiene sus ventajas y desventajas, pero en general, buscan medir la capacidad del modelo para capturar la información relevante y presentarla de manera clara y concisa.

**Archivo:** Evaluación de 